# Training Log Metrics Extraction and Visualization

This notebook extracts `loss`, `mse`, `mae`, `val_loss`, `val_mse`, and `val_mae` from:

- `output_lstmd.log.txt`
- `output_lstm_vast.log.txt`
- `output_logs_hbnn.log.txt`

It then renders separate training-curve plots for each file, compares model performance, and reports summary statistics.


## 1. Import Required Libraries


In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## 2. Load and Parse Training Logs


In [ ]:
ROOT = Path.cwd()
LOG_FILES = {
    "LSTM-D": ROOT / "output_lstmd.log.txt",
    "LSTM-VAST": ROOT / "output_lstm_vast.log.txt",
    "HBNN": ROOT / "output_logs_hbnn.log.txt",
}

epoch_pattern = re.compile(r"Epoch\s+(\d+)\s*/\s*(\d+)")
metric_pattern = re.compile(
    r"(val_loss|val_mse|val_mae|loss|mse|mae):\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)"
)
required_metrics = ["loss", "mse", "mae", "val_loss", "val_mse", "val_mae"]


def parse_log_file(log_path: Path) -> pd.DataFrame:
    if not log_path.exists():
        print(f"Warning: file not found -> {log_path}")
        return pd.DataFrame(columns=["run_id", "epoch", *required_metrics])

    current_epoch = None
    run_id = -1
    rows = []

    with log_path.open("r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            epoch_match = epoch_pattern.search(line)
            if epoch_match:
                current_epoch = int(epoch_match.group(1))
                if current_epoch == 1:
                    run_id += 1

            metric_pairs = metric_pattern.findall(line)
            if not metric_pairs or current_epoch is None:
                continue

            metric_dict = {}
            for key, value in metric_pairs:
                metric_dict[key] = float(value)

            if all(k in metric_dict for k in required_metrics):
                rows.append(
                    {
                        "run_id": run_id,
                        "epoch": current_epoch,
                        "loss": metric_dict["loss"],
                        "mse": metric_dict["mse"],
                        "mae": metric_dict["mae"],
                        "val_loss": metric_dict["val_loss"],
                        "val_mse": metric_dict["val_mse"],
                        "val_mae": metric_dict["val_mae"],
                    }
                )

    df = pd.DataFrame(rows)
    if df.empty:
        return pd.DataFrame(
            columns=["run_id", "epoch", *required_metrics, "global_step"]
        )

    df = df.sort_values(["run_id", "epoch"]).reset_index(drop=True)

    offsets = []
    cumulative = 0
    for rid in sorted(df["run_id"].unique()):
        run_count = (df["run_id"] == rid).sum()
        offsets.extend(list(np.arange(1, run_count + 1) + cumulative))
        cumulative += run_count

    df["global_step"] = offsets
    return df


metrics_by_model = {
    model_name: parse_log_file(path) for model_name, path in LOG_FILES.items()
}

## 3. Extract Training Metrics


In [ ]:
for model_name, df in metrics_by_model.items():
    print(f"\n{model_name}: parsed rows = {len(df)}")
    if df.empty:
        print("Warning: no metric rows found.")
        continue
    display(df.head())

## 4. Visualize Training Curves


In [ ]:
metric_pairs = [
    ("loss", "val_loss", "Loss"),
    ("mse", "val_mse", "MSE"),
    ("mae", "val_mae", "MAE"),
]

for model_name, df in metrics_by_model.items():
    if df.empty:
        print(f"Skipping plot for {model_name}: no extracted metrics.")
        continue

    fig, axes = plt.subplots(1, 3, figsize=(18, 4.5), sharex=True)
    fig.suptitle(
        f"{model_name} Training Metrics ({LOG_FILES[model_name].name})", fontsize=13
    )

    for ax, (train_col, val_col, title) in zip(axes, metric_pairs):
        ax.plot(df["global_step"], df[train_col], label=train_col, linewidth=1.8)
        ax.plot(
            df["global_step"], df[val_col], label=val_col, linewidth=1.8, linestyle="--"
        )
        ax.set_title(title)
        ax.set_xlabel("Global Step")
        ax.set_ylabel(title)
        ax.legend()

    plt.tight_layout()
    plt.show()

## 5. Compare Model Performance


In [ ]:
comparison_rows = []
for model_name, df in metrics_by_model.items():
    if df.empty:
        continue

    best_idx = df["val_loss"].idxmin()
    best_epoch = int(df.loc[best_idx, "epoch"])
    best_run = int(df.loc[best_idx, "run_id"])
    best_val_loss = float(df.loc[best_idx, "val_loss"])

    final_row = df.iloc[-1]

    comparison_rows.append(
        {
            "model": model_name,
            "best_val_loss": best_val_loss,
            "best_val_mse": float(df.loc[best_idx, "val_mse"]),
            "best_val_mae": float(df.loc[best_idx, "val_mae"]),
            "best_epoch": best_epoch,
            "best_run_id": best_run,
            "final_val_loss": float(final_row["val_loss"]),
            "final_val_mse": float(final_row["val_mse"]),
            "final_val_mae": float(final_row["val_mae"]),
            "total_points": int(len(df)),
        }
    )

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

if not comparison_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

    sns.barplot(data=comparison_df, x="model", y="best_val_loss", ax=axes[0])
    axes[0].set_title("Best Validation Loss")

    sns.barplot(data=comparison_df, x="model", y="best_val_mse", ax=axes[1])
    axes[1].set_title("Best Validation MSE")

    sns.barplot(data=comparison_df, x="model", y="best_val_mae", ax=axes[2])
    axes[2].set_title("Best Validation MAE")

    for ax in axes:
        ax.set_xlabel("Model")

    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(7, 4.5))
    sns.barplot(data=comparison_df, x="model", y="best_epoch")
    plt.title("Convergence Speed (Epoch of Best Val Loss)")
    plt.xlabel("Model")
    plt.ylabel("Epoch")
    plt.tight_layout()
    plt.show()
else:
    print("No data available for model comparison plots.")

## 6. Generate Summary Statistics


In [ ]:
summary_rows = []

for model_name, df in metrics_by_model.items():
    if df.empty:
        summary_rows.append(
            {
                "model": model_name,
                "runs": 0,
                "final_val_loss": np.nan,
                "best_val_loss": np.nan,
                "best_epoch": np.nan,
                "convergence_epoch_1pct": np.nan,
                "improvement_rate_pct": np.nan,
            }
        )
        continue

    runs = int(df["run_id"].nunique())
    final_val_loss = float(df.iloc[-1]["val_loss"])
    first_val_loss = float(df.iloc[0]["val_loss"])

    best_idx = df["val_loss"].idxmin()
    best_val_loss = float(df.loc[best_idx, "val_loss"])
    best_epoch = int(df.loc[best_idx, "epoch"])

    tolerance = max(abs(best_val_loss) * 0.01, 1e-12)
    threshold = best_val_loss + tolerance
    convergence_candidates = df.index[df["val_loss"] <= threshold]
    convergence_epoch = (
        int(df.loc[convergence_candidates[0], "epoch"])
        if len(convergence_candidates)
        else np.nan
    )

    improvement_rate_pct = (
        (first_val_loss - best_val_loss) / max(abs(first_val_loss), 1e-12)
    ) * 100.0

    summary_rows.append(
        {
            "model": model_name,
            "runs": runs,
            "final_val_loss": final_val_loss,
            "best_val_loss": best_val_loss,
            "best_epoch": best_epoch,
            "convergence_epoch_1pct": convergence_epoch,
            "improvement_rate_pct": improvement_rate_pct,
        }
    )

summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df.sort_values("model").reset_index(drop=True)
display(summary_df)

if not summary_df.empty:
    print("\nSummary (rounded):")
    display(summary_df.round(6))